# BAM Quality Control with pysam

Key QC metrics for a WGS BAM:
- **Mapping rate**: % reads that aligned to the reference
- **Duplication rate**: % reads flagged as PCR/optical duplicates
- **Coverage depth**: Mean depth across target regions
- **Coverage uniformity**: % bases at >=20x depth
- **Insert size distribution**: Reflects library fragment size
- **MAPQ distribution**: High MAPQ = confident unique alignments

Expected values for a 30x WGS Illumina library:
- Mapping rate: >95%
- Duplication rate: 5–20% (higher with lower input DNA)
- Mean coverage: ~30x
- % bases >=20x: >95%
- Insert size peak: 300–400 bp

In [ ]:
import pysam
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

BAM_PATH = "../results/alignment/NA12878.bqsr.bam"
REGION = "chr20"       # Query whole chr20; limit to a window for speed
REGION_START = 1000000
REGION_END   = 5000000

In [ ]:
# --- flagstat-equivalent metrics ---

FLAG_PAIRED     = 0x1
FLAG_PROPER     = 0x2
FLAG_UNMAPPED   = 0x4
FLAG_MATE_UNMAP = 0x8
FLAG_DUPLICATE  = 0x400
FLAG_SECONDARY  = 0x100
FLAG_SUPPLEMENTAL = 0x800

counts = Counter()

try:
    with pysam.AlignmentFile(BAM_PATH, 'rb') as bam:
        for read in bam.fetch(REGION, REGION_START, REGION_END):
            counts['total'] += 1
            if read.flag & FLAG_SECONDARY:    counts['secondary'] += 1
            if read.flag & FLAG_SUPPLEMENTAL: counts['supplemental'] += 1
            if read.flag & FLAG_DUPLICATE:    counts['duplicate'] += 1
            if read.flag & FLAG_UNMAPPED:     counts['unmapped'] += 1
            elif not (read.flag & FLAG_SECONDARY) and not (read.flag & FLAG_SUPPLEMENTAL):
                counts['mapped_primary'] += 1
            if read.flag & FLAG_PAIRED:       counts['paired'] += 1
            if read.flag & FLAG_PROPER:       counts['properly_paired'] += 1

    total = counts['total']
    print(f"Total reads:        {total:>10,}")
    print(f"Mapped (primary):   {counts['mapped_primary']:>10,}  ({100*counts['mapped_primary']/total:.1f}%)")
    print(f"Unmapped:           {counts['unmapped']:>10,}  ({100*counts['unmapped']/total:.1f}%)")
    print(f"Duplicates:         {counts['duplicate']:>10,}  ({100*counts['duplicate']/total:.1f}%)")
    print(f"Properly paired:    {counts['properly_paired']:>10,}  ({100*counts['properly_paired']/total:.1f}%)")
    print(f"Secondary:          {counts['secondary']:>10,}")
    print(f"Supplemental:       {counts['supplemental']:>10,}")

except FileNotFoundError:
    print(f"BAM not found: {BAM_PATH}. Run the pipeline first.")

In [ ]:
# --- Coverage depth using pysam.pileup ---

WINDOW_START = 1000000
WINDOW_END   = 1100000  # 100 kb window for speed

depths = []

try:
    with pysam.AlignmentFile(BAM_PATH, 'rb') as bam:
        for col in bam.pileup(REGION, WINDOW_START, WINDOW_END,
                               truncate=True, min_mapping_quality=20):
            depths.append(col.nsegments)

    depths = np.array(depths)
    print(f"Window: {REGION}:{WINDOW_START:,}-{WINDOW_END:,} ({len(depths):,} positions)")
    print(f"Mean depth:         {depths.mean():.1f}x")
    print(f"Median depth:       {np.median(depths):.1f}x")
    print(f"% bases >= 20x:     {(depths >= 20).mean()*100:.1f}%")
    print(f"% bases >= 10x:     {(depths >= 10).mean()*100:.1f}%")
    print(f"% zero coverage:    {(depths == 0).mean()*100:.1f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Depth histogram
    axes[0].hist(depths, bins=50, range=(0, depths.mean() * 3), color='steelblue')
    axes[0].axvline(x=depths.mean(), color='red', linestyle='--', label=f'Mean={depths.mean():.0f}x')
    axes[0].axvline(x=20, color='orange', linestyle='--', label='20x threshold')
    axes[0].set_title('Coverage depth distribution')
    axes[0].set_xlabel('Depth (x)')
    axes[0].legend()

    # Coverage over the window
    positions = np.arange(WINDOW_START, WINDOW_START + len(depths))
    axes[1].plot(positions[::100], depths[::100], color='steelblue', linewidth=0.5)
    axes[1].axhline(y=20, color='orange', linestyle='--', alpha=0.7, label='20x')
    axes[1].set_title(f'Coverage profile ({REGION}:{WINDOW_START//1000}k-{WINDOW_END//1000}k)')
    axes[1].set_xlabel('Genomic position')
    axes[1].set_ylabel('Depth')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"BAM not found: {BAM_PATH}")

In [ ]:
# --- Insert size distribution (paired-end fragment length) ---

insert_sizes = []
SAMPLE_LIMIT = 50000

try:
    with pysam.AlignmentFile(BAM_PATH, 'rb') as bam:
        for read in bam.fetch(REGION, REGION_START, REGION_END):
            # Only count properly paired, non-duplicate, primary reads on forward strand
            if (read.flag & 0x2 and  # properly paired
                not (read.flag & 0x400) and  # not duplicate
                not (read.flag & 0x100) and  # not secondary
                read.template_length > 0):   # forward read
                insert_sizes.append(read.template_length)
                if len(insert_sizes) >= SAMPLE_LIMIT:
                    break

    insert_sizes = np.array(insert_sizes)
    print(f"Insert size statistics ({len(insert_sizes):,} reads):")
    print(f"  Median: {np.median(insert_sizes):.0f} bp")
    print(f"  Mean:   {insert_sizes.mean():.0f} bp")
    print(f"  Std:    {insert_sizes.std():.0f} bp")
    print(f"  P10/P90: {np.percentile(insert_sizes, 10):.0f} / {np.percentile(insert_sizes, 90):.0f} bp")

    plt.figure(figsize=(10, 4))
    plt.hist(insert_sizes, bins=100, range=(0, 800), color='steelblue')
    plt.axvline(x=np.median(insert_sizes), color='red', linestyle='--',
                label=f'Median={np.median(insert_sizes):.0f} bp')
    plt.title('Insert size distribution')
    plt.xlabel('Insert size (bp)')
    plt.ylabel('Read count')
    plt.legend()
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"BAM not found: {BAM_PATH}")